# 05 — Eğitim (Faz 4b: baseline, Faz 5: deneyler)

**Çalışma zamanı: GPU.** Runtime → Change runtime type → L4 ya da A100 (T4, 640 için de yeterli ama yavaş).

Her deney bu notebook'la yapılır; sadece **Ayarlar** hücresindeki `EXP` adı ve değiştirilen tek parametre farklıdır. Test seti bu notebook'ta **hiç açılmaz**; deneyler sadece val'de karşılaştırılır.

**Akış**
1. Dataset (`DSET`, varsayılan `firesmoke_v2`) Drive'dan açılır.
2. **Duman testi:** train'in %10'uyla 2 kısa epoch. Pipeline'ın çalıştığı doğrulanır, tam eğitimin süresi ve işlem birimi maliyeti tahmin edilir.
3. **Tam eğitim:** ağırlıklar her epoch Drive'a yazılır. Oturum koparsa kaldığı yerden devam edilir.
4. **Sonuç:** val'de sınıf başına metrikler `reports/experiments.csv` deney günlüğüne eklenir.

Ultralytics sürümü `8.4.183`'te sabit: sürüm değişirse deneyler birbiriyle kıyaslanamaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!pip -q install ultralytics==8.4.183

In [ ]:
import os, json, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
RUNS   = ROOT / 'runs'

# --- Deney ayarları: her deneyde EXP adı + TEK bir değişiklik ----------------
DSET     = 'firesmoke_v2'     # datasets/<DSET>.zip
EXP      = 'E03_y11s_640_40ep'
MODEL    = 'yolo11s.pt'
IMGSZ    = 640
EPOCHS   = 40                 # E01'de 100 epoch'luk takvim öğrenme oranını geç düşürdü; 40 tam takvim
PATIENCE = 0                  # 0: erken durdurma kapalı (val gürültülü, takvim sonuna kadar gitsin)
BATCH    = -1                 # -1: GPU belleğinin ~%60'ına göre otomatik
CACHE    = 'disk'             # 'disk': ilk epoch'ta küçültülmüş kopyalar yazılır, sonrası hızlı
AUG      = dict(fliplr=0.5, flipud=0.0, degrees=0.0)   # gökyüzü yukarıda kalmalı (audit)
NOTE     = 'E01 + doğru takvim (40 epoch, erken durdurma kapalı)'
CU_PER_HOUR = None            # Runtime → View resources'taki saatlik birim; maliyet tahmini için
# ------------------------------------------------------------------------------

DS_ZIP = ROOT / 'datasets' / f'{DSET}.zip'
DATA   = Path('/content') / DSET
YAML   = DATA / 'data.yaml'

RUNS.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)
WORKERS = min(8, os.cpu_count() or 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('ultralytics', ultralytics.__version__, '· torch', torch.__version__, '·', GPU, '· CPU çekirdek:', os.cpu_count())
assert not (RUNS / EXP).exists() or (RUNS / EXP / 'weights' / 'last.pt').exists(), \
    f'{EXP} klasörü var ama last.pt yok — EXP adını değiştir'

## 1. Dataset'i aç

`DSET` Drive'dan `/content`'e açılır (~5–8 dk). Eğitim Drive'dan değil buradan okur.

In [ ]:
t0 = time.time()
if not YAML.exists():
    assert DS_ZIP.exists(), f'{DS_ZIP} yok'
    subprocess.run(['unzip', '-q', '-o', str(DS_ZIP), '-d', str(DATA.parent)], check=True)
man = pd.read_csv(DATA / 'manifest.csv')
counts = {s: sum(1 for p in (DATA / 'images' / s).iterdir() if p.suffix == '.jpg')   # cache .npy'leri sayma
          for s in ['train', 'val', 'test']}
print(counts, f'({time.time() - t0:.0f} sn)')
assert counts == man.split.value_counts().reindex(['train', 'val', 'test']).to_dict()
print(YAML.read_text())

## 2. Duman testi ve maliyet tahmini

Train'in %10'uyla 2 epoch koşar, val bu aşamada atlanır. Ardından val ayrıca zamanlanır. Tam epoch süresi ≈ 10 × (2. epoch süresi) + val süresi. 1. epoch önbellek yazdığı için hesaba katılmaz.

Bakılacaklar:
- Hata yok, loss değerleri sayı (NaN değil).
- `runs/<EXP>_smoke/train_batch0.jpg`: kutular doğru yerde mi?
- Tahmini süre ve birim. `CU_PER_HOUR`'ı **Runtime → View resources** ekranındaki saatlik tüketimle doldurursan maliyeti de hesaplar.

In [ ]:
stamps = []
def _epoch_end(trainer):
    stamps.append(time.time())

smoke_dir = RUNS / f'{EXP}_smoke'
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
sm = YOLO(MODEL)
sm.add_callback('on_train_epoch_end', _epoch_end)
sm.train(data=str(YAML), imgsz=IMGSZ, epochs=2, fraction=0.10, batch=BATCH, workers=WORKERS,
         cache=CACHE, val=False, plots=True, project=str(RUNS), name=smoke_dir.name,
         seed=0, deterministic=True, **AUG)
t_ep10 = stamps[-1] - stamps[-2] if len(stamps) >= 2 else stamps[-1] - stamps[0]

t0 = time.time()
sm.val(data=str(YAML), imgsz=IMGSZ, batch=16, workers=WORKERS, plots=False, verbose=False)
t_val = time.time() - t0

t_epoch = 10 * t_ep10 + t_val
print(f'\n%10 train epoch: {t_ep10:.0f} sn · val: {t_val:.0f} sn · tahmini tam epoch: {t_epoch / 60:.1f} dk')
for n_ep in [EPOCHS, 60, 40]:
    h = n_ep * t_epoch / 3600
    cost = f' · ~{h * CU_PER_HOUR:.0f} birim' if CU_PER_HOUR else ''
    print(f'{n_ep:>3} epoch: ~{h:.1f} saat{cost}')
print('\nPATIENCE nedeniyle eğitim genellikle EPOCHS dolmadan durur; bu üst sınır tahminidir.')

Süre ya da maliyet bütçeyi aşıyorsa **önce bana yaz**. Seçenekler: `EPOCHS`'u düşürmek, daha güçlü bir GPU seçmek ya da baseline'ı daha küçük bir modelle (`yolo11n.pt`) koşmak. Her biri sonuçları farklı etkiler.

## 3. Tam eğitim

Ağırlıklar (`last.pt`, `best.pt`) her epoch `runs/<EXP>/weights/` altına, Drive'a yazılır. **Oturum koparsa:** notebook'u baştan çalıştır (Drive bağlama, ayarlar, dataset), duman testini atla ve doğrudan bu hücreyi çalıştır. Hücre `last.pt`'yi görürse kaldığı yerden devam eder.

In [ ]:
last = RUNS / EXP / 'weights' / 'last.pt'

def finished(p):
    # Ultralytics eğitim bitince last.pt'yi sadeleştirir ve epoch'u -1 yapar
    return torch.load(p, map_location='cpu', weights_only=False).get('epoch', -1) == -1

t_start = time.time()
if last.exists() and finished(last):
    print('Bu deneyin eğitimi zaten tamamlanmış — 4. bölüme geç.')
elif last.exists():
    print('devam ediliyor:', last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(data=str(YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
                workers=WORKERS, cache=CACHE, project=str(RUNS), name=EXP, exist_ok=False,
                seed=0, deterministic=True, plots=True, **AUG)
train_hours = (time.time() - t_start) / 3600
print(f'eğitim süresi (bu oturum): {train_hours:.2f} saat')

## 4. Sonuçlar (val)

`best.pt` val setinde tekrar ölçülür. Sınıf başına metrikler ve eğitim eğrileri gösterilir, deney günlüğüne bir satır eklenir.

Eğrilerde bakılacaklar:
- **train ve val loss** birlikte düşüyor mu? Val loss yukarı dönüyorsa overfitting.
- **mAP50** hangi epoch'ta düzleşiyor? Erken düzleşme daha az epoch'un yeteceğini gösterir.

In [ ]:
run = RUNS / EXP
res = pd.read_csv(run / 'results.csv')
res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for c in ['train/box_loss', 'val/box_loss']:
    ax[0].plot(res.epoch, res[c], label=c)
for c in ['train/cls_loss', 'val/cls_loss']:
    ax[1].plot(res.epoch, res[c], label=c)
for c in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
    ax[2].plot(res.epoch, res[c], label=c)
for a in ax:
    a.legend(fontsize=8); a.set_xlabel('epoch')
plt.tight_layout(); plt.savefig(run / 'curves_custom.png', dpi=100); plt.show()

best = YOLO(str(run / 'weights' / 'best.pt'))
r = best.val(data=str(YAML), split='val', imgsz=IMGSZ, batch=16, workers=WORKERS, plots=True,
             project=str(run), name='val_best', exist_ok=True)
names = r.names
cls_idx = list(r.box.ap_class_index)
per_class = pd.DataFrame({
    'sınıf': [names[c] for c in cls_idx],
    'P': r.box.p, 'R': r.box.r, 'mAP50': r.box.ap50, 'mAP50-95': r.box.ap,
}).round(3)
display(per_class)

best_ep = int(res.loc[res['metrics/mAP50-95(B)'].idxmax(), 'epoch'])
row = {
    'tarih': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'), 'EXP': EXP, 'model': MODEL, 'imgsz': IMGSZ,
    'epoch_koşulan': int(res.epoch.max()), 'en_iyi_epoch': best_ep, 'batch': BATCH, 'cache': CACHE,
    'aug': json.dumps(AUG), 'not': NOTE, 'dataset': DSET, 'gpu': GPU, 'ultralytics': ultralytics.__version__,
    'mAP50': round(float(r.box.map50), 4), 'mAP50-95': round(float(r.box.map), 4),
    **{f'mAP50_{names[c]}': round(float(r.box.ap50[j]), 4) for j, c in enumerate(cls_idx)},
    **{f'mAP50-95_{names[c]}': round(float(r.box.ap[j]), 4) for j, c in enumerate(cls_idx)},
    **{f'R_{names[c]}': round(float(r.box.r[j]), 4) for j, c in enumerate(cls_idx)},
}
log = REP / 'experiments.csv'
df_log = pd.concat([pd.read_csv(log), pd.DataFrame([row])], ignore_index=True) if log.exists() else pd.DataFrame([row])
df_log = df_log.drop_duplicates(subset='EXP', keep='last')
df_log.to_csv(log, index=False)
display(df_log)

**Kontrol noktası — bana gönderilecekler:**
1. Duman testinin süre/maliyet satırları
2. Sınıf başına metrik tablosu ve deney günlüğü
3. Eğrilerin tarifi ya da ekran görüntüsü; ayrıca `runs/<EXP>/confusion_matrix_normalized.png`

Ardından aynı `EXP` ile `05b_diag`'ı çalıştır: kare bazında yakalama, sahne ortalaması ve train↔val farkı oradan gelir. Test setine Faz 6'ya kadar dokunulmuyor.